In [1]:
import gymnasium as gym
import numpy as np
import torch
from torch import nn

In [2]:
env = gym.make("CartPole-v1")

In [3]:
obs, info = env.reset()

## DQN

In [4]:
class miniDQN(nn.Module):
    def __init__(self, obs_dim, hidden_dim, num_action):
        super().__init__()
        self.linear_1 = nn.Linear(obs_dim, hidden_dim)
        self.linear_2 = nn.Linear(hidden_dim, hidden_dim)
        self.action_layer = nn.Linear(hidden_dim, num_action)
        
    def forward(self, x):
        x = self.linear_1(x)
        x = nn.ReLU()(x)
        x = self.linear_2(x)
        x = nn.GELU()(x)
        x = self.action_layer(x)

        return x

In [5]:
dqn_net = miniDQN(4, 16, 2)
target_network = miniDQN(4, 16, 2)
target_network.load_state_dict(dqn_net.state_dict())

<All keys matched successfully>

In [6]:
dqn_net(torch.tensor(obs))

tensor([-0.3314,  0.1447], grad_fn=<ViewBackward0>)

In [7]:
batch_size = 32
lr = 1e-4
optimizer = torch.optim.Adamax(dqn_net.parameters(), lr)
loss_fn = nn.SmoothL1Loss()# nn.MSELoss()

In [8]:
torch.argmax(dqn_net(torch.tensor(obs)))

tensor(1)

In [9]:
optimizer

Adamax (
Parameter Group 0
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)

In [10]:
replay_buffer = []
epsilon_greedy = 0.5

In [11]:
# q_values = dqn_net(torch.tensor(obs))

# if torch.randn(1).item() < epsilon_greedy:
#     action = env.action_space.sample()
# else:
#     action = q_values.argmax(dim=1).item()

# next_obs, reward, terminated, truncated, info = env.step(action)
# done = terminated or truncated


In [12]:
# replay_buffer.append((obs, action, reward, next_obs, done))

In [13]:
def play_game(replay_buffer, dqn_net, epsilon=0.5, training=True):
    obs, info = env.reset()
    done = False
    while not done:
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32)
        q_values = dqn_net(obs_tensor)

        if torch.rand(1).item() < epsilon:
            action = int(env.action_space.sample())
        else:
            action = q_values.argmax().item()

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.append((obs, action, reward, next_obs, done))
        obs = next_obs

    return replay_buffer

number_of_games = 2500000

# for game_no in range(number_of_games):
#     replay_buffer = play_game(epsilon_greedy, replay_buffer)


In [14]:
@torch.no_grad()
def soft_update(target_net, online_net, tau=0.05):
    for target_param, online_param in zip(target_net.parameters(), online_net.parameters()):
        target_param.mul_(1 - tau).add_(online_param, alpha=tau)

In [15]:
def dqn_train_step(batch, Q_network, target_network, optimizer, gamma=0.9):
    obs, actions, rewards, next_obs, dones = zip(*batch)

    obs = torch.as_tensor(np.stack(obs), dtype=torch.float32)
    actions = torch.as_tensor(actions, dtype=torch.long)
    rewards = torch.as_tensor(rewards, dtype=torch.float32)
    next_obs = torch.as_tensor(np.stack(next_obs), dtype=torch.float32)
    dones = torch.as_tensor(dones, dtype=torch.float32)

    q_values = Q_network(obs)
    q_taken = q_values.gather(1, actions[:, None]).squeeze(1)

    with torch.no_grad():
        max_q_next = target_network(next_obs).max(dim=1).values
        td_target = rewards + gamma * (1.0 - dones) * max_q_next

    loss = loss_fn(q_taken, td_target)
    optimizer.zero_grad()
    loss.backward()
    # torch.nn.utils.clip_grad_norm_(Q_network.parameters(), max_norm=1)
    optimizer.step()
    return loss.item()


In [16]:
# import random

# for step in range(number_of_games):
#     epsilon = max(0.05, 1.0 - step / 1000)
#     play_game(replay_buffer, dqn_net, epsilon)
#     if len(replay_buffer) < 1000:
#         continue
#     batch = random.sample(replay_buffer, batch_size)
#     loss = dqn_train_step(batch, dqn_net, target_network, optimizer)
#     soft_update(target_network, dqn_net, 0.09)

#     if step % 10 == 0:
#         print(f"step={step}, loss={loss:.4f}, buffer={len(replay_buffer)}")


In [17]:
new_replay_buffer = []
play_game(new_replay_buffer, dqn_net, 0)

[(array([-0.01534798, -0.04529369,  0.00234156,  0.02351096], dtype=float32),
  1,
  1.0,
  array([-0.01625385,  0.14979461,  0.00281178, -0.26843226], dtype=float32),
  False),
 (array([-0.01625385,  0.14979461,  0.00281178, -0.26843226], dtype=float32),
  1,
  1.0,
  array([-0.01325796,  0.34487632, -0.00255687, -0.560227  ], dtype=float32),
  False),
 (array([-0.01325796,  0.34487632, -0.00255687, -0.560227  ], dtype=float32),
  1,
  1.0,
  array([-0.00636043,  0.54003406, -0.01376141, -0.8537144 ], dtype=float32),
  False),
 (array([-0.00636043,  0.54003406, -0.01376141, -0.8537144 ], dtype=float32),
  1,
  1.0,
  array([ 0.00444025,  0.73534083, -0.03083569, -1.1506926 ], dtype=float32),
  False),
 (array([ 0.00444025,  0.73534083, -0.03083569, -1.1506926 ], dtype=float32),
  1,
  1.0,
  array([ 0.01914707,  0.93085134, -0.05384954, -1.4528831 ], dtype=float32),
  False),
 (array([ 0.01914707,  0.93085134, -0.05384954, -1.4528831 ], dtype=float32),
  1,
  1.0,
  array([ 0.03776409

In [18]:
len(new_replay_buffer)

10

In [19]:
grup = [(1, 2, 3), (4, 5, 6)]

In [20]:
list(zip(*grup))

[(1, 4), (2, 5), (3, 6)]

In [21]:
list(zip((1, 2, 3), (4, 5, 6)))

[(1, 4), (2, 5), (3, 6)]

In [22]:
import torch

# q_values: (Batch Size, Action Size) -> 3 örnek, 4 aksiyon
q_values = torch.tensor([
    [10, 12, 15, 11], # 0. örnek için Q değerleri
    [20, 25, 22, 21], # 1. örnek için Q değerleri
    [ 5,  8,  7,  9]  # 2. örnek için Q değerleri
])

# actions: Hangi aksiyonların seçildiği (indeksler)
actions = torch.tensor([2, 0, 3])
print(actions.shape)

torch.Size([3])


In [23]:
actions[:, None], actions[:, None].shape

(tensor([[2],
         [0],
         [3]]),
 torch.Size([3, 1]))

In [24]:
q_values.gather(1, actions[:, None]).squeeze(1)

tensor([15, 20,  9])

In [25]:
q_values.mul_(2)

tensor([[20, 24, 30, 22],
        [40, 50, 44, 42],
        [10, 16, 14, 18]])

In [26]:
q_values

tensor([[20, 24, 30, 22],
        [40, 50, 44, 42],
        [10, 16, 14, 18]])

In [27]:
q_values.add_(5, alpha=2)

tensor([[30, 34, 40, 32],
        [50, 60, 54, 52],
        [20, 26, 24, 28]])

In [28]:
from torch.distributions import Categorical
import torch


In [29]:
probs = torch.tensor([0.1, 0.3, 0.6])

dist = Categorical(probs=probs)

sample = dist.sample((3,))
print(sample)


tensor([2, 1, 2])


In [30]:
dist.entropy()

tensor(0.8979)

In [31]:
dist.log_prob(torch.tensor([2]))

tensor([-0.5108])

In [32]:
sample

tensor([2, 1, 2])

In [33]:
dist.log_prob(sample)

tensor([-0.5108, -1.2040, -0.5108])

## Policy Based Methods

In [34]:
env = gym.make("CartPole-v1")
obs, info = env.reset()

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

device = "cpu"

class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return F.softmax(x, dim=1)
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample() # for training to explore, torch.argmax(probs, dim=-1) othervise
        return action.item(), m.log_prob(action)

In [36]:
obs = torch.tensor(obs)
obs.shape

torch.Size([4])

In [37]:
linear_1 = nn.Linear(obs.shape[0], 16)
linear_2 = nn.Linear(16, env.action_space.n )

In [38]:
x = linear_1(obs)
x = linear_2(x)

In [39]:
x

tensor([ 0.2360, -0.2133], grad_fn=<ViewBackward0>)

In [40]:
p = F.softmax(x, dim=-1)
p

tensor([0.6105, 0.3895], grad_fn=<SoftmaxBackward0>)

In [41]:
m = Categorical(p)

In [42]:
m.probs, m.logits

(tensor([0.6105, 0.3895], grad_fn=<DivBackward0>),
 tensor([-0.4935, -0.9428], grad_fn=<LogBackward0>))

## Pendulum via Policy

In [43]:
env = gym.make("Pendulum")
obs, info = env.reset()

/Users/doruksarigun/Kaggle/orbit-wars/orbit_wars_env/lib/python3.14/site-packages/gymnasium/envs/registration.py:520: UserWarning: WARN: Using the latest versioned environment `Pendulum-v1` instead of the unversioned environment `Pendulum`.
  logger.warn(


In [44]:
obs

array([ 3.34514268e-02, -9.99440372e-01,  1.08417036e-04], dtype=float32)

In [45]:
list(env.__dict__.keys())[2].endswith("space")

True

In [46]:
spaces_to_consider = [space for space in list(env.__dict__.keys()) if space.endswith("space")]

In [47]:
for space in spaces_to_consider:
    print(f"{space}: -> {getattr(env, space[1:])}")

_action_space: -> Box(-2.0, 2.0, (1,), float32)
_observation_space: -> Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)


In [48]:
s_size = env.observation_space.shape[0]
a_size = env.action_space.shape[0]
action_low = env.action_space.low
action_high = env.action_space.high

In [49]:
s_size, a_size, action_high, action_low

(3, 1, array([2.], dtype=float32), array([-2.], dtype=float32))

In [50]:
getattr(env, space)

In [51]:
print(f"Observation space: {env.observation_space}")
print()

Observation space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)



In [52]:
from torch.distributions import Normal

device = "cpu"

class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)
        self.log_std = nn.Parameter(torch.zeros(a_size) * -0.5)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        mean = self.fc2(x)
        
        return torch.tanh(mean) * float(action_high[0])
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        mean = self.forward(state)
        std = torch.exp(self.log_std).expand_as(mean)
        dist = Normal(mean, std)
        raw_action = dist.sample()
        squashed_action = torch.tanh(raw_action)

        action_scale = torch.tensor(
            (action_high - action_low) / 2.0,
            dtype=torch.float32
        ).to(device)

        action_bias = torch.tensor(
            (action_high + action_low) / 2.0,
            dtype=torch.float32
        ).to(device)

        action = squashed_action * action_scale + action_bias

        action_clamped = torch.clamp(
            raw_action,
            torch.tensor(action_low, dtype=torch.float32).to(device),
            torch.tensor(action_high, dtype=torch.float32).to(device)
        )
        # log_prob = dist.log_prob(raw_action).sum(dim=-1)

        log_prob = dist.log_prob(raw_action).sum(dim=-1)

        log_prob -= torch.log(
            action_scale * (1 - squashed_action.pow(2)) + 1e-6
        ).sum(dim=-1)


        #return action_clamped.cpu().numpy()[0], log_prob
        return action.cpu().numpy()[0], log_prob
    

state, _ = env.reset()
policy = Policy(s_size, a_size, 64).to(device)

action, log_prob = policy.act(state)
# action, raw_action, squashed_action = policy.act(state)

action, type(action), action.shape, log_prob, log_prob.shape
# action, raw_action, squashed_action


(array([-1.7639242], dtype=float32),
 numpy.ndarray,
 (1,),
 tensor([-0.4285], grad_fn=<SubBackward0>),
 torch.Size([1]))

In [53]:
state_torch = torch.from_numpy(state).float().unsqueeze(0).to(device)
mean = policy(state_torch)
mean

tensor([[-0.5835]], grad_fn=<MulBackward0>)

In [54]:
policy.log_std

Parameter containing:
tensor([-0.], requires_grad=True)

In [55]:
std = torch.exp(policy.log_std).expand_as(mean)
std

tensor([[1.]], grad_fn=<ExpandBackward0>)

In [56]:
dist = Normal(mean, std)
dist.sample()

tensor([[-1.0570]])

In [57]:
mean

tensor([[-0.5835]], grad_fn=<MulBackward0>)

In [58]:
sampled_action = dist.sample()
print(sampled_action)
dist.log_prob(torch.tensor([-0.1964]))

tensor([[0.0605]])


tensor([[-0.9939]], grad_fn=<SubBackward0>)

In [59]:
dir(dist)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_batch_shape',
 '_event_shape',
 '_extended_shape',
 '_get_checked_instance',
 '_log_normalizer',
 '_mean_carrier_measure',
 '_natural_params',
 '_validate_args',
 '_validate_sample',
 'arg_constraints',
 'batch_shape',
 'cdf',
 'entropy',
 'enumerate_support',
 'event_shape',
 'expand',
 'has_enumerate_support',
 'has_rsample',
 'icdf',
 'loc',
 'log_prob',
 'mean',
 'mode',
 'perplexity',
 'rsample',
 'sample',
 'sample_n',
 'scale',
 'set_default_validate_args',
 'stddev',
 'support',
 'variance']

In [60]:
action, log_prob = policy.act(state)

loss = -log_prob.mean()
policy.zero_grad()
loss.backward()

policy.log_std.grad


tensor([0.6639])

In [61]:
action

array([-1.6442196], dtype=float32)

In [62]:
def collect_trajectory(env, policy, max_t=200):
    saved_log_probs = []
    rewards = []
    state, _ = env.reset()

    for t in range(max_t):
        action, log_prob = policy.act(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        saved_log_probs.append(log_prob)
        rewards.append(reward)

        state = next_state

        if done:
            break

    return saved_log_probs, rewards

saved_log_probs, rewards = collect_trajectory(env, policy)

len(saved_log_probs), len(rewards), sum(rewards)



(200, 200, np.float64(-1374.448788250883))

In [63]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0

    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32).to(device)
    return returns

returns = compute_returns(rewards)

returns.shape, returns[:5], returns.mean(), returns.std()

(torch.Size([200]),
 tensor([-546.5038, -549.7503, -552.7267, -555.1022, -556.4366]),
 tensor(-416.7050),
 tensor(160.6741))

In [64]:
def normalize_returns(returns):
    return (returns - returns.mean()) / (returns.std() + 1e-8)

norm_returns = normalize_returns(returns)

norm_returns.shape, norm_returns[:5], norm_returns.mean(), norm_returns.std()

(torch.Size([200]),
 tensor([-0.8078, -0.8280, -0.8466, -0.8614, -0.8697]),
 tensor(-1.1384e-07),
 tensor(1.0000))

In [65]:
def compute_policy_loss(saved_log_probs, returns):
    policy_loss = []

    for log_prob, G in zip(saved_log_probs, returns):
        policy_loss.append(-log_prob * G)

    return torch.cat(policy_loss).sum()

loss = compute_policy_loss(saved_log_probs, norm_returns)

loss, loss.shape

(tensor(-9.1177, grad_fn=<SumBackward0>), torch.Size([]))

In [66]:
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)

In [67]:
saved_log_probs, rewards = collect_trajectory(env, policy)

returns = compute_returns(rewards)
norm_returns = normalize_returns(returns)

loss = compute_policy_loss(saved_log_probs, norm_returns)

optimizer.zero_grad()
loss.backward()
optimizer.step()

sum(rewards), loss.item()

(np.float64(-1643.4107229267966), 5.699556350708008)

In [68]:
def train_one_episode(env, policy, optimizer, gamma=0.99):
    saved_log_probs, rewards = collect_trajectory(env, policy)

    returns = compute_returns(rewards, gamma)
    returns = normalize_returns(returns)

    loss = compute_policy_loss(saved_log_probs, returns)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return sum(rewards), loss.item()

In [69]:
episode_reward, loss_value = train_one_episode(env, policy, optimizer)

episode_reward, loss_value

(np.float64(-1446.7614555959253), -49.58118438720703)

In [70]:
def train(env, policy, optimizer, n_episodes=500, gamma=0.99, print_every=100):
    scores = []

    for episode in range(1, n_episodes + 1):
        episode_reward, loss_value = train_one_episode(
            env, policy, optimizer, gamma
        )

        scores.append(episode_reward)

        if episode % print_every == 0:
            avg_score = np.mean(scores[-print_every:])
            print(
                f"Episode {episode}\tAverage Score: {avg_score:.2f}\tLoss: {loss_value:.2f}"
            )

    return scores


# scores = train(env, policy, optimizer, n_episodes=50_000, print_every=100)

In [71]:
policy.log_std.data, torch.exp(policy.log_std.data)

(tensor([9.5792e-05]), tensor([1.0001]))

In [72]:
liste = []
liste.insert(0, 1)

In [73]:
liste.insert(0, 2)

In [74]:
liste

[2, 1]

In [75]:
class Critic(nn.Module):
    def __init__(self, s_size, h_size):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        value = self.fc2(x)

        return value


In [76]:
critic = Critic(s_size, 64).to(device)
state, _ = env.reset()

state_t = torch.from_numpy(state).float().unsqueeze(0).to(device)

value = critic(state_t)
value, value.shape


(tensor([[0.0792]], grad_fn=<AddmmBackward0>), torch.Size([1, 1]))

In [77]:
def collect_trajectory_ac(env, policy, critic, max_t=200):
    saved_log_probs = []
    values = []
    rewards = []

    state, _ = env.reset()

    for t in range(max_t):
        state_t = torch.from_numpy(state).float().unsqueeze(0).to(device)
        value = critic(state_t)

        action, log_prob = policy.act(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        saved_log_probs.append(log_prob)
        values.append(value.squeeze(-1))
        rewards.append(reward)

        state = next_state

        if done:
            break
    
    return saved_log_probs, values, rewards

saved_log_probs, values, rewards = collect_trajectory_ac(env, policy, critic)

len(saved_log_probs), len(values), len(rewards), values[0].shape


(200, 200, 200, torch.Size([1]))

In [78]:
returns = compute_returns(rewards)
returns = normalize_returns(returns)

values_t = torch.cat(values)

returns.shape, values_t.shape

(torch.Size([200]), torch.Size([200]))

In [81]:
advantages = returns - values_t
advantages.shape, advantages[:5], advantages.mean(), advantages.std()

(torch.Size([200]),
 tensor([-0.4503, -0.4627, -0.4981, -0.5244, -0.5412], grad_fn=<SliceBackward0>),
 tensor(0.6391, grad_fn=<MeanBackward0>),
 tensor(1.0379, grad_fn=<StdBackward0>))

In [82]:
values_t[:5], returns[:5]

(tensor([-0.3080, -0.3261, -0.3216, -0.3264, -0.3411], grad_fn=<SliceBackward0>),
 tensor([-0.7583, -0.7888, -0.8197, -0.8508, -0.8823]))

In [83]:
log_probs_t = torch.cat(saved_log_probs)
actor_loss = -(log_probs_t * advantages.detach()).sum()

In [84]:
log_probs_t.shape, advantages.shape, actor_loss, actor_loss.shape

(torch.Size([200]),
 torch.Size([200]),
 tensor(39.4186, grad_fn=<NegBackward0>),
 torch.Size([]))

In [85]:
critic_loss = F.mse_loss(values_t, returns)
critic_loss, critic_loss.shape

(tensor(1.4802, grad_fn=<MseLossBackward0>), torch.Size([]))

In [86]:
total_loss = actor_loss + critic_loss * 0.5

In [87]:
actor_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

In [88]:
actor_optimizer.zero_grad()
critic_optimizer.zero_grad()

total_loss.backward()

actor_optimizer.step()
critic_optimizer.step()

total_loss.item(), actor_loss.item(), critic_loss.item()

(40.15867614746094, 39.418582916259766, 1.4801844358444214)

In [90]:
def train_one_episode_ac(env, policy, critic, actor_optimizer, critic_optimizer, gamma=0.99):
    saved_log_probs, values, rewards = collect_trajectory_ac(env, policy, critic)

    returns = compute_returns(rewards, gamma)
    returns = normalize_returns(returns)

    values_t = torch.cat(values)
    log_probs_t = torch.cat(saved_log_probs)

    advantages = returns - values_t

    actor_loss = -(log_probs_t * advantages.detach()).mean()
    critic_loss = F.mse_loss(values_t, returns)

    total_loss = actor_loss + 0.5 * critic_loss

    actor_optimizer.zero_grad()
    critic_optimizer.zero_grad()

    total_loss.backward()

    actor_optimizer.step()
    critic_optimizer.step()

    return sum(rewards), actor_loss.item(), critic_loss.item()


In [93]:
episode_reward, actor_loss_value, critic_loss_value = train_one_episode_ac(
    env, policy, critic, actor_optimizer, critic_optimizer
)

episode_reward, actor_loss_value, critic_loss_value


(np.float64(-1627.9845263980806), -0.16084276139736176, 0.9745875000953674)

In [94]:
policy = Policy(s_size, a_size, 64).to(device)
critic = Critic(s_size, 64).to(device)

actor_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

def train_ac(
    env,
    policy,
    critic,
    actor_optimizer,
    critic_optimizer,
    n_episodes=5000,
    gamma=0.99,
    print_every=100,
):
    scores = []

    for episode in range(1, n_episodes + 1):
        episode_reward, actor_loss_value, critic_loss_value = train_one_episode_ac(
            env,
            policy,
            critic,
            actor_optimizer,
            critic_optimizer,
            gamma,
        )

        scores.append(episode_reward)

        if episode % print_every == 0:
            avg_score = np.mean(scores[-print_every:])
            print(
                f"Episode {episode}\t"
                f"Average Score: {avg_score:.2f}\t"
                f"Actor Loss: {actor_loss_value:.3f}\t"
                f"Critic Loss: {critic_loss_value:.3f}"
            )

    return scores


scores_ac = train_ac(
    env,
    policy,
    critic,
    actor_optimizer,
    critic_optimizer,
    n_episodes=30_000,
    print_every=100,
)

Episode 100	Average Score: -1390.52	Actor Loss: 0.088	Critic Loss: 0.881
Episode 200	Average Score: -1394.99	Actor Loss: -0.076	Critic Loss: 0.976
Episode 300	Average Score: -1397.59	Actor Loss: 0.137	Critic Loss: 0.689
Episode 400	Average Score: -1367.68	Actor Loss: -0.078	Critic Loss: 0.914
Episode 500	Average Score: -1431.94	Actor Loss: 0.002	Critic Loss: 0.850
Episode 600	Average Score: -1453.36	Actor Loss: -0.253	Critic Loss: 0.826
Episode 700	Average Score: -1430.64	Actor Loss: -0.009	Critic Loss: 0.852
Episode 800	Average Score: -1415.81	Actor Loss: -0.085	Critic Loss: 0.730
Episode 900	Average Score: -1415.07	Actor Loss: 0.034	Critic Loss: 0.971
Episode 1000	Average Score: -1408.34	Actor Loss: 0.029	Critic Loss: 0.827
Episode 1100	Average Score: -1412.46	Actor Loss: -0.191	Critic Loss: 0.618
Episode 1200	Average Score: -1418.29	Actor Loss: 0.066	Critic Loss: 0.658
Episode 1300	Average Score: -1414.60	Actor Loss: -0.016	Critic Loss: 1.136
Episode 1400	Average Score: -1403.65	Act

In [95]:
policy.log_std.data, torch.exp(policy.log_std.data)

(tensor([-0.4770]), tensor([0.6207]))

In [96]:
np.mean(scores_ac[-100:]), np.max(scores_ac[-100:])

(np.float64(-293.43858646369983), np.float64(-0.8633283133150643))

In [97]:
state, _ = env.reset()

state_t = torch.from_numpy(state).float().unsqueeze(0).to(device)
value = critic(state_t)

action, log_prob = policy.act(state)

next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

next_state_t = torch.from_numpy(next_state).float().unsqueeze(0).to(device)

with torch.no_grad():
    next_value = critic(next_state_t)

reward_t = torch.tensor([reward], dtype=torch.float32).to(device)
done_t = torch.tensor([done], dtype=torch.float32).to(device)

target = reward_t + 0.99 * next_value.squeeze(-1) * (1 - done_t)
td_error = target - value.squeeze(-1)

actor_loss = -(log_prob * td_error.detach()).mean()
critic_loss = td_error.pow(2).mean()

actor_loss, critic_loss, td_error

(tensor(-0.0899, grad_fn=<NegBackward0>),
 tensor(0.0196, grad_fn=<MeanBackward0>),
 tensor([-0.1400], grad_fn=<SubBackward0>))

In [101]:
def train_one_episode_td_ac(
        env,
        policy,
        critic,
        actor_optimizer,
        critic_optimizer,
        gamma=0.99,
        max_t=200,
        reward_scale=10.0
):
    actor_losses = []
    critic_losses = []
    rewards = []

    state, _ = env.reset()

    for i in range(max_t):
        state_t = torch.from_numpy(state).float().unsqueeze(0).to(device)
        value = critic(state_t)

        action, log_prob = policy.act(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        next_state_t = torch.from_numpy(next_state).float().unsqueeze(0).to(device)

        with torch.no_grad():
            next_value = critic(next_state_t)

        # reward_t = torch.tensor([reward], dtype=torch.float32).to(device)
        reward_t = torch.tensor([reward / reward_scale], dtype=torch.float32).to(device)
        done_t = torch.tensor([done], dtype=torch.float32).to(device)

        target = reward_t + gamma * next_value.squeeze(-1) * (1 - done_t)
        td_error = target - value.squeeze(-1)

        actor_loss = -(log_prob * td_error.detach()).mean()
        critic_loss = td_error.pow(2).mean()

        actor_losses.append(actor_loss)
        critic_losses.append(critic_loss)
        rewards.append(reward)

        state = next_state

        if done: break

    actor_loss = torch.stack(actor_losses).mean()
    critic_loss = torch.stack(critic_losses).mean()
    total_loss = actor_loss + 0.5 * critic_loss

    actor_optimizer.zero_grad()
    critic_optimizer.zero_grad()

    total_loss.backward()

    actor_optimizer.step()
    critic_optimizer.step()

    return sum(rewards), actor_loss.item(), critic_loss.item()


policy = Policy(s_size, a_size, 64).to(device)
critic = Critic(s_size, 64).to(device)

actor_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)


episode_reward, actor_loss_value, critic_loss_value = train_one_episode_td_ac(
    env, policy, critic, actor_optimizer, critic_optimizer
)

episode_reward, actor_loss_value, critic_loss_value


(np.float64(-1392.4298343297403), 0.2665834426879883, 0.7599119544029236)

In [102]:
def train_td_ac(
    env,
    policy,
    critic,
    actor_optimizer,
    critic_optimizer,
    n_episodes=5000,
    gamma=0.99,
    print_every=100,
):
    scores = []

    for episode in range(1, n_episodes + 1):
        episode_reward, actor_loss_value, critic_loss_value = train_one_episode_td_ac(
            env,
            policy,
            critic,
            actor_optimizer,
            critic_optimizer,
            gamma,
        )

        scores.append(episode_reward)

        if episode % print_every == 0:
            avg_score = np.mean(scores[-print_every:])
            print(
                f"Episode {episode}\t"
                f"Average Score: {avg_score:.2f}\t"
                f"Actor Loss: {actor_loss_value:.3f}\t"
                f"Critic Loss: {critic_loss_value:.3f}"
            )

    return scores

In [103]:
policy = Policy(s_size, a_size, 64).to(device)
critic = Critic(s_size, 64).to(device)

actor_optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)

scores_td_ac = train_td_ac(
    env,
    policy,
    critic,
    actor_optimizer,
    critic_optimizer,
    n_episodes=30_000,
    print_every=100,
)

Episode 100	Average Score: -1614.82	Actor Loss: -1.117	Critic Loss: 0.785
Episode 200	Average Score: -1744.47	Actor Loss: -1.079	Critic Loss: 0.929
Episode 300	Average Score: -1707.81	Actor Loss: -0.673	Critic Loss: 1.977
Episode 400	Average Score: -1736.62	Actor Loss: -0.925	Critic Loss: 4.350
Episode 500	Average Score: -1584.63	Actor Loss: 0.173	Critic Loss: 7.959
Episode 600	Average Score: -1463.64	Actor Loss: -0.246	Critic Loss: 6.421
Episode 700	Average Score: -1437.42	Actor Loss: -0.115	Critic Loss: 8.480
Episode 800	Average Score: -1452.19	Actor Loss: 0.101	Critic Loss: 12.013
Episode 900	Average Score: -1387.86	Actor Loss: -0.434	Critic Loss: 8.108
Episode 1000	Average Score: -1395.33	Actor Loss: -0.556	Critic Loss: 9.140
Episode 1100	Average Score: -1351.18	Actor Loss: 0.040	Critic Loss: 4.001
Episode 1200	Average Score: -1182.87	Actor Loss: -0.202	Critic Loss: 9.133
Episode 1300	Average Score: -1224.28	Actor Loss: -0.194	Critic Loss: 8.529
Episode 1400	Average Score: -1214.95